# Phase 3 v2 — Kaggle Inference (vLLM Ensemble + Majority Vote)

Runs 3 fine-tuned SLMs with vLLM and combines predictions via character-level majority voting.

| Model | Size | tp | JSON valid % |
|-------|------|----|-------------|
| `Qwen/Qwen3-1.7B` | 1.7B | 1 | 100% |
| `Qwen/Qwen3-8B` | 8B | 2 | 95% |
| `LiquidAI/LFM2.5-1.2B-Instruct` | 1.2B | 1 | 100% |

**Hardware**: T4 x2 on Kaggle (internet OFF)

| | |
|---|---|
| **Wheels** | from `0-kaggle-setup-nbme-score-clinical-pip-wheel_v2` dataset |
| **Models** | from `0-kaggle-setup-nbme-score-clinical-pip-wheel_v2` dataset |
| **Adapters** | from `adapter-nbme-score-clinical` dataset |
| **Output** | `/kaggle/working/submission.csv` |


In [1]:
import os
import sys
import shutil
import subprocess
import site
from pathlib import Path

# =========================
# CONFIG
# =========================
WHEELS = "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels"
PKG_DIR = "/kaggle/working/pkgs"

# Same package specs — pip finds the matching wheels in WHEELS
PACKAGES = [
    "tokenizers>=0.22.0,<=0.23.0",   # must match transformers==4.56.0 requirement; Kaggle base may have wrong version
    "transformers==4.56.0",
    "vllm==0.17.1",
    "rapidfuzz>=3.0.0",
    "protobuf<6",
    "huggingface-hub>=0.34.0,<1.0",
    "msgspec>=0.18.0",
    "peft>=0.15.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", WHEELS,
    *PACKAGES,
])

# =========================
# VERIFY
# =========================
import torch
import transformers
import vllm
import rapidfuzz
import huggingface_hub
import msgspec
import google.protobuf

def where(mod):
    """Show where a module is loaded from."""
    return getattr(mod, "__file__", "builtin/namespace")

print("\n==== VERSION CHECK ====")
print(f"torch:            {torch.__version__}")
print(f"transformers:     {transformers.__version__}  ({where(transformers)})")
print(f"vllm:             {vllm.__version__}  ({where(vllm)})")
print(f"rapidfuzz:        {rapidfuzz.__version__}  ({where(rapidfuzz)})")
print(f"huggingface_hub:  {huggingface_hub.__version__}  ({where(huggingface_hub)})")
print(f"msgspec:          {msgspec.__version__}  ({where(msgspec)})")
print(f"protobuf:         {google.protobuf.__version__}  ({where(google.protobuf)})")

print("\n==== CUDA ====")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GB | CC {p.major}.{p.minor}")

# =========================
# GPU GUARD — fail fast
# =========================
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n*** NO GPU DETECTED ***\n"
        "Go to Kaggle Settings → Accelerator → GPU T4 x2 → Save, then re-run.\n"
    )
n_gpus = torch.cuda.device_count()
if n_gpus < 2:
    print(f"\n⚠  WARNING: only {n_gpus} GPU detected. Qwen3-8B uses tp=2 and needs T4 x2.")
    print("   If this is intentional (e.g. debug run), set qwen3_8b tp=1 in MODEL_REGISTRY.")
else:
    total_vram = sum(
        torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)
    ) / 1024**3
    print(f"\n✓ {n_gpus} GPUs detected — total VRAM: {total_vram:.1f} GB")

Looking in links: /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/transformers-4.56.0-py3-none-any.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/vllm-0.17.1-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/rapidfuzz-3.14.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/huggingface_hub-0.36.2-py3-none-any.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels/bi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.



==== VERSION CHECK ====
torch:            2.10.0+cu128
transformers:     4.56.0  (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
vllm:             0.17.1  (/usr/local/lib/python3.12/dist-packages/vllm/__init__.py)
rapidfuzz:        3.14.5  (/usr/local/lib/python3.12/dist-packages/rapidfuzz/__init__.py)
huggingface_hub:  0.36.2  (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)
msgspec:          0.21.1  (/usr/local/lib/python3.12/dist-packages/msgspec/__init__.py)
protobuf:         5.29.6  (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)

==== CUDA ====
CUDA available: True
GPU 0: Tesla T4 | 14.6 GB | CC 7.5
GPU 1: Tesla T4 | 14.6 GB | CC 7.5

✓ 2 GPUs detected — total VRAM: 29.1 GB


In [2]:
import contextlib, gc, json, logging, re, shutil, sys, time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from rapidfuzz.fuzz import partial_ratio_alignment
from tqdm import tqdm
from transformers import AutoTokenizer

from vllm import LLM, SamplingParams
from vllm.config import AttentionConfig
from vllm.v1.attention.backends.registry import AttentionBackendEnum
from vllm.sampling_params import StructuredOutputsParams
from vllm.lora.request import LoRARequest

try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    def destroy_model_parallel(): pass

print(f"vLLM version: {__import__('vllm').__version__}")

# T4 CC=7.5 — no native bfloat16
_DTYPE = torch.float16

# Wall-clock start for elapsed tracking
_WALL_START = time.time()

def elapsed_str() -> str:
    s = time.time() - _WALL_START
    return f"{s/3600:.2f}h / 9.00h elapsed"

CONFIG = {
    "DATA_DIR":              Path("/kaggle/input/competitions/nbme-score-clinical-patient-notes"),
    "OUTPUT_DIR":            Path("/kaggle/working"),
    "GPU_MEM_UTIL":          0.90,
    "MAX_MODEL_LEN":         1024,
    "MAX_NEW_TOKENS":        256,   # worst-case output ~88 tokens; 256 is safe headroom
    "LLM_TEMPERATURE":       0.0,
    "MAX_SPANS_PER_FEATURE": 10,
    "VOTE_THRESHOLD":        2,
    "FUZZY_SCORE_CUTOFF":    70.0,
    "SEED":                  42,
    "LORA_RANK":             16,    # matches LORA_R=16 in 2_train_slm_kaggle_compatible_group1.ipynb
    # Debug: set to e.g. 50 to run on subset only (estimates per-row speed, checks timeout risk)
    # Set to None for full competition run.
    "DEBUG_N":               None,
}

MODEL_REGISTRY = [
    {
        "name":         "qwen3_1_7b",
        "model_path":   "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b",
        "vllm_dtype":   "half",
        "tp":           1,
        "adapter_path": "/kaggle/input/datasets/natbrian/adapter-nbme-score-clinical-v2/adapters/qwen3_1_7b_adapter",
        "enable_thinking": False,
        "trust_remote_code": False,
    },
    {
        "name":         "qwen3_8b",
        "model_path":   "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-8b/models/qwen3_8b",
        "vllm_dtype":   "half",
        "tp":           2,
        "adapter_path": "/kaggle/input/datasets/natbrian/adapter-nbme-score-clinical-v2/adapters/qwen3_8b_adapter",
        "enable_thinking": False,
        "trust_remote_code": False,
    },
    {
        "name":         "lfm2_5_1_2b",
        "model_path":   "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-lfm2-5-1-2b/models/lfm2_5_1_2b",
        "vllm_dtype":   "half",
        "tp":           1,
        "adapter_path": "/kaggle/input/datasets/natbrian/adapter-nbme-score-clinical-v2/adapters/lfm2_5_1_2b_adapter",
        "enable_thinking": None,
        "trust_remote_code": True,
    },
]

SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    '{"spans": ["exact text 1", "exact text 2"]}'
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)
print("✓ CONFIG, MODEL_REGISTRY loaded")
print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
print(f"  DEBUG_N: {CONFIG['DEBUG_N']} (None = full run)")

2026-05-02 06:20:02.715208: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777702803.109623      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777702803.226853      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777702804.234393      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777702804.234442      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777702804.234445      22 computation_placer.cc:177] computation placer alr

vLLM version: 0.17.1
✓ CONFIG, MODEL_REGISTRY loaded
  Models: ['qwen3_1_7b', 'qwen3_8b', 'lfm2_5_1_2b']
  DEBUG_N: None (None = full run)


## Section 1 — Per-Note Regex FSM Constraint

In [3]:
def _build_char_class(note_chars: set) -> str:
    parts = []
    for ch in sorted(note_chars, key=ord):
        code = ord(ch)
        if code < 0x20 or code == 0x7F: continue
        if ch == ']':    parts.append(r'\]')
        elif ch == '^':  parts.append(r'\^')
        elif ch == '-':  parts.append(r'\-')
        elif ch == '\\': parts.append(r'\\')
        else:            parts.append(ch)
    return '[' + ''.join(parts) + ']' if parts else r'[^\n]'


def build_constraint_regex(pn_history: str, max_spans: int = 10) -> str:
    note_chars = set(pn_history) - {'"', '\\'}
    char_class = _build_char_class(note_chars)
    span_item  = f'"{char_class}*"'
    additional = r'(?:, ' + span_item + r'){0,' + str(max_spans - 1) + r'}'
    opt_list   = r'(?:' + span_item + additional + r')?'
    return r'\{"spans": \[' + opt_list + r'\]\}'

print("✓ Section 1: build_constraint_regex defined")


✓ Section 1: build_constraint_regex defined


## Section 2 — vLLM Engine Lifecycle (native LoRA, no merge)

In [4]:
def init_engine(model_spec: dict, cfg: dict) -> LLM:
    attn_cfg = AttentionConfig(backend=AttentionBackendEnum.TRITON_ATTN)
    log.info(f"  [{model_spec['name']}] Initialising vLLM engine (tp={model_spec['tp']}) ...")
    llm = LLM(
        model                  = model_spec["model_path"],
        dtype                  = model_spec["vllm_dtype"],
        tensor_parallel_size   = model_spec["tp"],
        gpu_memory_utilization = cfg["GPU_MEM_UTIL"],
        max_model_len          = cfg["MAX_MODEL_LEN"],
        enforce_eager          = True,   # T4 CC=7.5: Triton shared-mem OOM without eager (vLLM#36802)
        enable_lora            = True,
        max_lora_rank          = cfg["LORA_RANK"],
        # enable_prefix_caching disabled: corruption bug with lora in 0.17.1 (vLLM#30931)
        trust_remote_code      = model_spec.get("trust_remote_code", False),
        seed                   = cfg["SEED"],
        attention_config       = attn_cfg,
    )
    log.info(f"  [{model_spec['name']}] vLLM engine ready.")
    return llm


def destroy_engine(llm, model_name: str) -> None:
    log.info(f"  [{model_name}] Destroying vLLM engine ...")
    destroy_model_parallel()
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(f"  [{model_name}] VRAM after cleanup: {free_gb:.1f}/{total_gb:.1f} GB free")

print("✓ Section 2: init_engine, destroy_engine defined")


✓ Section 2: init_engine, destroy_engine defined


## Section 3 — Prompt Builder

In [5]:
def build_chat_prompt(feature_text: str, pn_history: str, tokenizer,
                      enable_thinking=None) -> str:
    # User content matches training exactly — no /no_think in content.
    # enable_thinking=False passed via apply_chat_template kwargs (Qwen3 template handles it).
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Note: \"{pn_history.strip()}\"\n"
            f"Feature: {feature_text}"
        )},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            pass
    return tokenizer.apply_chat_template(messages, **kwargs)

print("✓ Section 3: build_chat_prompt defined")


✓ Section 3: build_chat_prompt defined


## Section 4 — vLLM Inference Runner

In [6]:
def _parse_json_output(raw_text: str) -> list:
    raw_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(raw_text)
        return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError):
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
            except json.JSONDecodeError:
                pass
    return []


def run_inference_vllm(llm, test_rows, pn_map, feat_map, tokenizer,
                       cfg, model_spec) -> list:
    model_name      = model_spec["name"]
    enable_thinking = model_spec.get("enable_thinking")
    lora_request    = LoRARequest(model_name, 1, model_spec["adapter_path"])

    log.info(f"  [{model_name}] Building prompts ...")
    prompts, params_list = [], []

    for _, row in test_rows.iterrows():
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        prompts.append(build_chat_prompt(feature_text, pn_history, tokenizer, enable_thinking))
        regex = build_constraint_regex(pn_history, cfg["MAX_SPANS_PER_FEATURE"])
        params_list.append(SamplingParams(
            temperature=cfg["LLM_TEMPERATURE"],
            max_tokens=cfg["MAX_NEW_TOKENS"],
            structured_outputs=StructuredOutputsParams(regex=regex),
        ))

    log.info(f"  [{model_name}] Running vLLM inference on {len(prompts)} rows ...")
    outputs   = llm.generate(prompts=prompts, sampling_params=params_list,
                              lora_request=lora_request)
    all_spans = []
    for output in tqdm(outputs, desc=f"  [{model_name}] Parsing", leave=False):
        raw = output.outputs[0].text.strip() if output.outputs else ""
        all_spans.append(_parse_json_output(raw))

    n_nonempty = sum(1 for s in all_spans if s)
    log.info(f"  [{model_name}] Done — non-empty: {n_nonempty}/{len(all_spans)}")
    return all_spans

print("✓ Section 4: _parse_json_output, run_inference_vllm defined")


✓ Section 4: _parse_json_output, run_inference_vllm defined


## Section 5 — Character-Level Majority Voting

In [7]:
def spans_to_char_array(span_locations: list, note_len: int) -> np.ndarray:
    arr = np.zeros(note_len, dtype=np.uint8)
    for start, end in span_locations:
        arr[max(0, start):min(note_len, end)] = 1
    return arr


def char_array_to_spans(arr: np.ndarray) -> list:
    spans, n, i = [], len(arr), 0
    while i < n:
        if arr[i] == 1:
            start = i
            while i < n and arr[i] == 1: i += 1
            spans.append((start, i))
        else:
            i += 1
    return spans


def locate_span_in_note(span_text: str, pn_history: str,
                        score_cutoff: float = 70.0) -> Optional[tuple]:
    span_text = span_text.strip()
    if not span_text or not pn_history: return None
    idx = pn_history.find(span_text)
    if idx != -1: return (idx, idx + len(span_text))
    idx = pn_history.lower().find(span_text.lower())
    if idx != -1: return (idx, idx + len(span_text))
    result = partial_ratio_alignment(span_text, pn_history, score_cutoff=score_cutoff)
    if result is not None: return (result.dest_start, result.dest_end)
    return None


def character_level_majority_vote(model_predictions, test_rows, pn_map,
                                  vote_threshold=2, fuzzy_cutoff=70.0) -> list:
    n_models, n_rows = len(model_predictions), len(test_rows)
    log.info(f"Majority vote ({n_models} models, threshold={vote_threshold}/{n_models}) ...")
    final_spans = []

    for seq_idx, (_, row) in enumerate(tqdm(test_rows.iterrows(), total=n_rows, desc="Majority vote")):
        pn_history = pn_map.get(row["pn_num"], "")
        note_len   = len(pn_history)
        if note_len == 0:
            final_spans.append([]); continue

        vote_array = np.zeros(note_len, dtype=np.int8)
        for preds in model_predictions:
            locs = [loc for text in preds[seq_idx]
                    if (loc := locate_span_in_note(text, pn_history, fuzzy_cutoff)) is not None]
            if locs:
                vote_array += spans_to_char_array(locs, note_len)

        consensus = (vote_array >= vote_threshold).astype(np.uint8)
        for i, ch in enumerate(pn_history):
            if ch in (' ', '\t', '\n', '\r') and consensus[i]:
                is_start = (i == 0 or consensus[i-1] == 0)
                is_end   = (i == note_len-1 or consensus[i+1] == 0)
                if is_start or is_end: consensus[i] = 0

        final_spans.append(char_array_to_spans(consensus))

    log.info(f"Vote complete — non-empty: {sum(1 for s in final_spans if s)}/{n_rows}")
    return final_spans

print("✓ Section 5: majority vote defined")


✓ Section 5: majority vote defined


## Section 6 — Submission Formatter

In [8]:
def format_location_string(spans: list, pn_history: str) -> str:
    if not spans: return ""
    clean = []
    for start, end in sorted(spans):
        while start < end and pn_history[start] in (' ', '\t', '\n', '\r'): start += 1
        while end > start and pn_history[end-1] in (' ', '\t', '\n', '\r'): end -= 1
        if start < end: clean.append((start, end))
    merged = []
    for start, end in sorted(clean):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return ";".join(f"{s} {e}" for s, e in merged) if merged else ""


def build_submission(final_spans: list, test_df: pd.DataFrame, pn_map: dict) -> pd.DataFrame:
    rows = []
    for row_idx, (_, test_row) in enumerate(test_df.iterrows()):
        pn_history = pn_map.get(test_row["pn_num"], "")
        spans      = final_spans[row_idx] if row_idx < len(final_spans) else []
        location   = format_location_string(spans, pn_history)
        rows.append({"id": test_row["id"], "location": location if location else np.nan})
    return pd.DataFrame(rows)

print("✓ Section 6: format_location_string, build_submission defined")


✓ Section 6: format_location_string, build_submission defined


## Run Phase 3 — Generate Submission

Pipeline per model:
1. Merge LoRA adapter into base weights (float16, device_map=auto)
2. Load merged model into vLLM with TRITON_ATTN backend
3. Run batched inference with per-note regex constrained decoding
4. Destroy engine + delete merged model from disk

Then: character-level majority vote → `submission.csv`


In [9]:
def main():
    cfg      = CONFIG
    data_dir = cfg["DATA_DIR"]
    debug_n  = cfg.get("DEBUG_N")

    print("\n" + "="*65)
    print("  PHASE 3 v2: Kaggle Inference (vLLM + native LoRA)")
    print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
    if debug_n:
        print(f"  *** DEBUG MODE: first {debug_n} rows only ***")
    print(f"  {elapsed_str()}")
    print("="*65 + "\n")

    print("▶ Loading test data ...")
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = feat_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()

    if debug_n:
        test_df = test_df.head(debug_n).copy()
        print(f"  ⚠ DEBUG: truncated to {len(test_df)} rows")
    print(f"  Test rows: {len(test_df)}")

    all_model_predictions = []
    timing_summary = []

    for i, model_spec in enumerate(MODEL_REGISTRY):
        model_name = model_spec["name"]
        print(f"\n{'='*65}")
        print(f"  Model {i+1}/{len(MODEL_REGISTRY)}: {model_name}  (tp={model_spec['tp']})")
        print(f"  Base:    {model_spec['model_path']}")
        print(f"  Adapter: {model_spec['adapter_path']}")
        print(f"  {elapsed_str()}")
        print(f"{'='*65}")

        t_model_start = time.time()

        # Load tokenizer from base model
        tokenizer = AutoTokenizer.from_pretrained(
            model_spec["model_path"],
            use_fast=True,
            trust_remote_code=model_spec.get("trust_remote_code", False),
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        t_engine_start = time.time()
        llm = init_engine(model_spec, cfg)
        t_engine_ready = time.time()
        print(f"  [{model_name}] Engine init: {t_engine_ready - t_engine_start:.1f}s")

        t_infer_start = time.time()
        model_spans = run_inference_vllm(llm, test_df, pn_map, feat_map,
                                         tokenizer, cfg, model_spec)
        t_infer_end = time.time()

        n_rows      = len(test_df)
        infer_secs  = t_infer_end - t_infer_start
        rows_per_s  = n_rows / infer_secs if infer_secs > 0 else 0
        print(f"  [{model_name}] Inference:   {infer_secs:.1f}s  ({rows_per_s:.2f} rows/s)")
        if debug_n:
            full_rows   = len(pd.read_csv(data_dir / "test.csv")) if debug_n else n_rows
            est_full_s  = full_rows / rows_per_s if rows_per_s > 0 else float("inf")
            print(f"  [{model_name}] Est. full run: ~{est_full_s:.0f}s ({est_full_s/60:.1f} min)")

        all_model_predictions.append(model_spans)

        destroy_engine(llm, model_name)
        t_model_end = time.time()
        model_total = t_model_end - t_model_start
        print(f"  [{model_name}] Total (init+infer+destroy): {model_total:.1f}s  |  {elapsed_str()}")
        timing_summary.append((model_name, model_total, infer_secs))

        del llm, tokenizer
        gc.collect()

    print("\n" + "="*65)
    print("  TIMING SUMMARY")
    print("="*65)
    for name, total, infer in timing_summary:
        print(f"  {name:20s}  total={total:.0f}s  inference={infer:.0f}s")
    print(f"  {elapsed_str()}")
    print("="*65)

    print("\n▶ Running character-level majority vote ...")
    effective_threshold = min(cfg["VOTE_THRESHOLD"], len(all_model_predictions))
    final_spans = character_level_majority_vote(
        all_model_predictions, test_df, pn_map,
        vote_threshold=effective_threshold,
        fuzzy_cutoff=cfg["FUZZY_SCORE_CUTOFF"],
    )

    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    print("\n" + "="*65)
    print(f"  ✓ Submission saved → {out_path}")
    print(f"  Shape: {submission_df.shape}")
    print(f"  Non-empty: {submission_df['location'].notna().sum()} / {len(submission_df)}")
    print(f"  {elapsed_str()}")
    print("="*65)
    print(submission_df.head(10).to_string())

main()


  PHASE 3 v2: Kaggle Inference (vLLM + native LoRA)
  Models: ['qwen3_1_7b', 'qwen3_8b', 'lfm2_5_1_2b']
  0.00h / 9.00h elapsed

▶ Loading test data ...
  Test rows: 5

  Model 1/3: qwen3_1_7b  (tp=1)
  Base:    /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b
  Adapter: /kaggle/input/datasets/natbrian/adapter-nbme-score-clinical-v2/adapters/qwen3_1_7b_adapter
  0.00h / 9.00h elapsed
INFO 05-02 06:20:33 [utils.py:238] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=True, use_trtllm_attention=None, disable_flashinfer_prefill=False, disable_f

2026-05-02 06:21:13.912383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777702873.939222      92 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777702873.947602      92 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777702873.966394      92 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777702873.966425      92 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777702873.966428      92 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=92) INFO 05-02 06:21:21 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b', speculative_config=None, tokenizer='/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), obser

[W502 06:21:23.395913609 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore_DP0 pid=92) INFO 05-02 06:21:24 [base.py:106] Offloader set to NoopOffloader
(EngineCore_DP0 pid=92) INFO 05-02 06:21:24 [gpu_model_runner.py:4281] Starting to load model /kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b...
(EngineCore_DP0 pid=92) INFO 05-02 06:21:25 [cuda.py:368] Using AttentionBackendEnum.TRITON_ATTN backend.


...

  [lfm2_5_1_2b] Inference:   7.1s  (0.71 rows/s)
  [lfm2_5_1_2b] Total (init+infer+destroy): 78.4s  |  0.12h / 9.00h elapsed


[rank0]:[W502 06:27:52.271354740 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
nanobind: leaked 2 instances!
 - leaked instance 0x7fbedc486fb8 of type "xgrammar.xgrammar_bindings.CompiledGrammar"
 - leaked instance 0x7fbebc166928 of type "xgrammar.xgrammar_bindings.GrammarMatcher"
nanobind: leaked 6 types!
 - leaked type "xgrammar.xgrammar_bindings.CompiledGrammar"
 - leaked type "xgrammar.xgrammar_bindings.TokenizerInfo"
 - leaked type "xgrammar.xgrammar_bindings.Grammar"
 - leaked type "xgrammar.xgrammar_bindings.GrammarMatcher"
 - leaked type "xgrammar.xgrammar_bindings.BatchGrammarMatcher"
 - leaked type "xgrammar.xgrammar_bindings.GrammarCompiler"
nanobind: leaked 51 functions!
 - leaked function ""
 - leaked function ""
 - leaked function "batch_fill_next_token_bitmask"
 - leaked function "co


  TIMING SUMMARY
  qwen3_1_7b            total=119s  inference=32s
  qwen3_8b              total=235s  inference=33s
  lfm2_5_1_2b           total=78s  inference=7s
  0.12h / 9.00h elapsed

▶ Running character-level majority vote ...


Majority vote: 100%|██████████| 5/5 [00:00<00:00, 119.16it/s]


  ✓ Submission saved → /kaggle/working/submission.csv
  Shape: (5, 2)
  Non-empty: 5 / 5
  0.12h / 9.00h elapsed
          id location
0  00016_000  696 724
1  00016_001  669 693
2  00016_002  203 217
3  00016_003    70 91
4  00016_004  222 258
